In [1]:
PROJECT_ID = "sabs-1000"
print("PROJECT_ID:", PROJECT_ID)

PROJECT_ID: sabs-1000


In [2]:
# Step 1 — Environment & Tools\n
!terraform -version\n
!gcloud version\n

Usage: terraform [global options] <subcommand> [args]

The available commands for execution are listed below.
The primary workflow commands are given first, followed by
less common or more advanced commands.

Main commands:
  init          Prepare your working directory for other commands
  validate      Check whether the configuration is valid
  plan          Show changes required by the current configuration
  apply         Create or update infrastructure
  destroy       Destroy previously-created infrastructure

All other commands:
  console       Try Terraform expressions at an interactive command prompt
  fmt           Reformat your configuration in the standard style
  force-unlock  Release a stuck lock on the current workspace
  get           Install or upgrade remote Terraform modules
  graph         Generate a Graphviz graph of the steps in an operation
  import        Associate existing infrastructure with a Terraform resource
  login         Obtain and save credentials for a

ERROR: (gcloud) Invalid choice: 'version\n'.
Maybe you meant:
  gcloud version

To search the help text of gcloud commands, run:
  gcloud help -- SEARCH_TERMS


In [4]:
# Clear any manually set credentials path\n
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = ""
print("Cleared GOOGLE_APPLICATION_CREDENTIALS")

Cleared GOOGLE_APPLICATION_CREDENTIALS


In [ ]:
# Step 2 — Authentication & Project Setup

# 1. Clear old sessions
!gcloud auth application-default revoke

# 2. Login fresh (may open a browser window)
!gcloud auth login
!gcloud auth application-default login

# 3. Force focus to target project
!gcloud config set project $PROJECT_ID
!gcloud auth application-default set-quota-project $PROJECT_ID

In [ ]:
# 4. Verification (The "Big Three")
!gcloud auth list
!gcloud config get-value project

import os, json, pathlib

creds_path = os.path.join(os.environ.get("APPDATA", ""), "gcloud", "application_default_credentials.json")
print("ADC path:", creds_path)

if os.path.exists(creds_path):
    with open(creds_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    text = json.dumps(data, indent=2)
    print(text[:2000])
else:
    print("application_default_credentials.json not found")

In [ ]:
# Step 3 — Enable Required APIs\n
!gcloud services enable artifactregistry.googleapis.com --project $PROJECT_ID\n
!gcloud services enable bigquery.googleapis.com --project $PROJECT_ID\n
!gcloud services enable iam.googleapis.com --project $PROJECT_ID\n

In [ ]:
# Step 4 — Prepare Terraform\n
import os\n
\n
os.chdir("terraform")\n
print("Current working directory:", os.getcwd())\n
\n
# Wipe old state (prevents cross-project issues)\n
for fname in ["terraform.tfstate", "terraform.tfstate.backup"]:\n
    if os.path.exists(fname):\n
        os.remove(fname)\n
        print(f"Removed {fname}")\n
    else:\n
        print(f"{fname} not found")\n
\n
# Fresh initialization\n
!terraform init -reconfigure\n

In [ ]:
# Step 5 — Plan\n
!terraform plan -var-file="dev.tfvars"\n

In [ ]:
# Step 6 — Apply\n
!terraform apply -var-file="dev.tfvars"\n

In [ ]:
# Done — infra deployed and variables recap\n
\n
artifact_repo_url = "asia-south1-docker.pkg.dev/dn-project-template-488407/sabs-dev-repo"\n
artifacts_bucket = "sabs-dev-artifacts-dn-project-template-488407"\n
cicd_service_account = "sabs-dev-cicd@dn-project-template-488407.iam.gserviceaccount.com"\n
data_bucket = "sabs-dev-data-dn-project-template-488407"\n
dataset_id = "training_dataset_dev"\n
project_id = "dn-project-template-488407"\n
runtime_service_account = "sabs-dev-runtime@dn-project-template-488407.iam.gserviceaccount.com"\n
table_id = "features_table"\n
\n
for name, value in [\n
    ("artifact_repo_url", artifact_repo_url),\n
    ("artifacts_bucket", artifacts_bucket),\n
    ("cicd_service_account", cicd_service_account),\n
    ("data_bucket", data_bucket),\n
    ("dataset_id", dataset_id),\n
    ("project_id", project_id),\n
    ("runtime_service_account", runtime_service_account),\n
    ("table_id", table_id),\n
]:\n
    print(f"{name}: {value}")\n